In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
# 1. CẤU HÌNH MÔI TRƯỜNG
MY_JAVA_HOME = r"D:\java\openjdk-17.0.18b8"
MY_HADOOP_HOME = r"D:\java\hadoop-3.4.3"
os.environ["JAVA_HOME"] = MY_JAVA_HOME
os.environ["HADOOP_HOME"] = MY_HADOOP_HOME
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
sys.path.append(os.path.join(MY_HADOOP_HOME, "bin"))
# 2. KHỞI TẠO SPARK SESSION
spark = SparkSession.builder \
    .appName('MetroPT3_SQL_Analysis_Group16') \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://10.125.222.18:9000") \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()
print("-> Trạng thái: Spark Session cho SQL đã sẵn sàng!")
# 3. ĐỌC DỮ LIỆU SẠCH VÀ TẠO ĐẶC TRƯNG THỜI GIAN
HDFS_CLEAN_FOR_SQL = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
print("-> Đang nạp dữ liệu Parquet từ HDFS...")
df = spark.read.parquet(HDFS_CLEAN_FOR_SQL)
# Cột 'timestamp' đã là dạng thời gian chuẩn, chỉ cần trích xuất thẳng ra giờ, tháng, ngày
df = df.withColumn('hour',    F.hour('timestamp')) \
       .withColumn('month',   F.month('timestamp')) \
       .withColumn('weekday', F.dayofweek('timestamp')) \
       .withColumn('date',    F.to_date('timestamp'))
# 4. LƯU TRỮ VÀO BỘ NHỚ ĐỆM (CACHE) & TẠO VIEW TRUY VẤN
print("-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...")
df.persist(StorageLevel.MEMORY_AND_DISK)
# Kích hoạt hành động (Trigger) để ép Spark thực sự nạp dữ liệu vào RAM
total_rows = df.count()
print(f"-> Cached thành công: {total_rows:,} rows")
# Tạo bảng ảo tên là 'sensor' để sử dụng cú pháp spark.sql()
df.createOrReplaceTempView('sensor')
print("=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===")

-> Trạng thái: Spark Session cho SQL đã sẵn sàng!
-> Đang nạp dữ liệu Parquet từ HDFS...
-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...
-> Cached thành công: 1,516,948 rows
=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===


In [2]:
q_startup_current = spark.sql("""
WITH startup_events AS (
    SELECT
        timestamp,
        Motor_current,
        COMP,
        LAG(COMP,1,1) OVER (ORDER BY timestamp) AS prev_comp
    FROM sensor
)
SELECT
    ROUND(AVG(Motor_current),4) AS dong_dien_khoi_dong_tb,
    ROUND(MIN(Motor_current),4) AS dong_dien_khoi_dong_min,
    ROUND(MAX(Motor_current),4) AS dong_dien_khoi_dong_max,
    COUNT(*) AS so_lan_khoi_dong
FROM startup_events
WHERE prev_comp = 1
  AND COMP = 0
""")
q_startup_current.show()

+----------------------+-----------------------+-----------------------+----------------+
|dong_dien_khoi_dong_tb|dong_dien_khoi_dong_min|dong_dien_khoi_dong_max|so_lan_khoi_dong|
+----------------------+-----------------------+-----------------------+----------------+
|                3.6043|                   0.02|                  9.295|           16923|
+----------------------+-----------------------+-----------------------+----------------+



In [3]:
# Q7: PHÁT HIỆN DÒNG ĐIỆN KHỞI ĐỘNG BẤT THƯỜNG
q7 = spark.sql('''
    SELECT *
    FROM (
        SELECT
            timestamp, date, hour, COMP,
            Motor_current AS dong_dien_luc_khoi_dong,
            LAG(COMP, 1, 1) OVER (ORDER BY timestamp) AS prev_comp,
            LEAD(Motor_current, 1) OVER (ORDER BY timestamp) AS dong_dien_sau_10s,
            LEAD(Motor_current, 2) OVER (ORDER BY timestamp) AS dong_dien_sau_20s,
            LEAD(Motor_current, 3) OVER (ORDER BY timestamp) AS dong_dien_sau_30s,
            GREATEST(
                Motor_current,
                LEAD(Motor_current, 1) OVER (ORDER BY timestamp),
                LEAD(Motor_current, 2) OVER (ORDER BY timestamp),
                LEAD(Motor_current, 3) OVER (ORDER BY timestamp)
            ) AS dong_dien_dinh
        FROM sensor)
    WHERE dong_dien_luc_khoi_dong > 0
      AND prev_comp = 1
      AND COMP = 0
    ORDER BY dong_dien_dinh DESC
    LIMIT 15''')
print("\n========== EXECUTION PLAN ==========\n")
q7.explain(True)
df_q7 = q7.toPandas()
print("--- TOP 15 LẦN KHỞI ĐỘNG CÓ DÒNG ĐIỆN ĐỈNH CAO NHẤT ---")
try:    display(df_q7)
except NameError:    print(df_q7.to_string(index=False))


========== EXECUTION PLAN ==========

== Parsed Logical Plan ==
'GlobalLimit 15
+- 'LocalLimit 15
   +- 'Sort ['dong_dien_dinh DESC NULLS LAST], true
      +- 'Project [*]
         +- 'Filter ((('dong_dien_luc_khoi_dong > 0) AND ('prev_comp = 1)) AND ('COMP = 0))
            +- 'SubqueryAlias __auto_generated_subquery_name
               +- 'Project ['timestamp, 'date, 'hour, 'COMP, 'Motor_current AS dong_dien_luc_khoi_dong#1293, 'LAG('COMP, 1, 1) windowspecdefinition('timestamp ASC NULLS FIRST, unspecifiedframe$()) AS prev_comp#1294, 'LEAD('Motor_current, 1) windowspecdefinition('timestamp ASC NULLS FIRST, unspecifiedframe$()) AS dong_dien_sau_10s#1295, 'LEAD('Motor_current, 2) windowspecdefinition('timestamp ASC NULLS FIRST, unspecifiedframe$()) AS dong_dien_sau_20s#1296, 'LEAD('Motor_current, 3) windowspecdefinition('timestamp ASC NULLS FIRST, unspecifiedframe$()) AS dong_dien_sau_30s#1297, 'GREATEST('Motor_current, 'LEAD('Motor_current, 1) windowspecdefinition('timestamp ASC NULLS

,timestamp,date,hour,COMP,dong_dien_luc_khoi_dong,prev_comp,dong_dien_sau_10s,dong_dien_sau_20s,dong_dien_sau_30s,dong_dien_dinh
0,2020-03-03 23:00:42,2020-03-03,23,0,9.2950,1,5.2575,5.6125,5.6925,9.2950
1,2020-05-16 14:17:03,2020-05-16,14,0,9.2700,1,5.1450,5.4575,5.6400,9.2700
2,2020-04-22 07:09:35,2020-04-22,7,0,9.1750,1,5.3250,5.9850,5.9300,9.1750
3,2020-04-06 04:19:35,2020-04-06,4,0,9.1600,1,5.5075,5.9325,5.8475,9.1600
4,2020-03-28 01:46:13,2020-03-28,1,0,9.1100,1,5.2850,5.9250,5.9950,9.1100
5,2020-03-15 10:55:48,2020-03-15,10,0,9.1000,1,5.1175,5.8925,5.8375,9.1000
6,2020-03-04 10:25:04,2020-03-04,10,0,9.0750,1,5.5100,5.7500,5.8850,9.0750
7,2020-06-10 09:59:35,2020-06-10,9,0,8.9200,1,5.1225,5.7200,5.9175,8.9200
8,2020-07-14 21:43:07,2020-07-14,21,0,8.8475,1,5.4375,5.8400,5.9425,8.8475
9,2020-05-17 08:42:23,2020-05-17,8,0,8.8250,1,5.1575,5.7900,5.8075,8.8250
